In [129]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
# import scipy.stats as st
import numpy as np
# import statannotations
import scipy
import statsmodels.api as sm
from pathlib import Path
from collections import defaultdict

# Get metabolomics data

In [69]:
repo_folder = Path('../../..')
data_folder  = repo_folder / 'data'/ 'this_project' / '6_transporterKO'
metabolomics_folder = data_folder / 'D_big_screen'


In [70]:
normalization = 'TIC'#
fn1 = metabolomics_folder / f'D_batch_1_4_{normalization}_norm.csv'
df1 = pd.read_csv(fn1)
df1.rename(columns={'metabolite':'Metabolite'}, inplace = True)

In [71]:
fn5 = metabolomics_folder / f'E_batch_5_{normalization}_norm.csv'
df5 = pd.read_csv(fn5)
df5.rename(columns={'metabolite':'Metabolite'}, inplace = True)

# Get KEGG IDs
Use KEGG IDs obtained from KEGG directly or iML1515, but skip those from biocyc, seems to contain a lot of strange metabolites

In [72]:
ecoli_kegg_ids = pd.read_csv(data_folder / 'Z_div/C_all_ecoli_kegg_ids.csv', index_col=0)['KEGG ID'].values

# Remove all metabolites that doesn't map to the E. coli metabolome kegg ids


### df1 batch 1-4

In [73]:
# bool_index = np.zeros(len(df), dtype = bool)
keep_idx = []
discarded_mets = []
for i, row in df1.iterrows():
    kegg_annotations = [x.strip(' ') for x in row['KEGG'].split(';')]
    mapped_kegg_annotations = [x for x in kegg_annotations if x in ecoli_kegg_ids]
    if len(mapped_kegg_annotations):
        keep_idx.append(i)
    else:
        discarded_mets.append(row['Metabolite'])

In [107]:
discarded_mets

['Acrolein',
 'Potassium acetate',
 'Butynal',
 'Propynoate',
 'Butynol',
 'Acrylamide',
 'Chloroethanol',
 'Hydroxybutynal',
 'Cyclopentanone',
 'Succinic aldehyde',
 'C5:2',
 'Succinimide',
 'Succinic anhydride',
 'Valerolactone',
 'Valerolactone',
 'Cyclohexanol',
 'N-Formiminoglycine',
 'Pentanoic acid',
 'NDEA',
 'Methylthioglycolic acid',
 'Diethylene glycol',
 'Nitrosobenzene',
 'Chloropropanoic acid',
 'Benzoquinone',
 'Benzosemiquinone',
 'Aminophenol',
 'Methyl methanesulfonic acid',
 'N-Acetylimidazole',
 '2,5-Dihydroxypyridine',
 'Monomethyl sulfate',
 'Hydroxymethylphosphonate',
 'Furoic acid',
 'Azomycin',
 'Acetylenedicarboxylic acid',
 '2-Hydroxycyclohexan-1-one',
 'Nitrosopiperidine',
 'Maleamic acid',
 'Diacetylhydrazine',
 'Aminopentanamide',
 'Guanidinoacetic acid',
 'Hydroxymalonic acid',
 'Tetrose',
 'Dimethylsulfonioacetate',
 'Benzamide',
 'Tromethamine',
 '3-Mercaptolactate',
 'Pyrazinamide',
 'Anisidine',
 'Methanearsonous acid',
 'N-Ethylmaleimide',
 'Formyl-

In [74]:
df1 = df1.loc[keep_idx]

### df5 batch 5

In [75]:
# bool_index = np.zeros(len(df), dtype = bool)
keep_idx = []
discarded_mets = []
for i, row in df5.iterrows():
    kegg_annotations = [x.strip(' ') for x in row['kegg'].split(';')]
    mapped_kegg_annotations = [x for x in kegg_annotations if x in ecoli_kegg_ids]
    if len(mapped_kegg_annotations):
        keep_idx.append(i)
    else:
        discarded_mets.append(row['Metabolite'])
df5 = df5.loc[keep_idx]

# Parse metadata

In [76]:
samples_fn = metabolomics_folder / 'A_samples.csv'
samples_df = pd.read_csv(samples_fn, index_col=0)

# Parse header
And make long dataframe with metadata

## Batch 1-4

In [77]:
# Optional parsing [x.replace(' ','').split('|') for x in df_norm.columns[3:]]
metadata1 = pd.DataFrame([[y.strip(' ') for y in x.split('|')] for x in df1.columns[3:]], columns = ['#/Batch', 'Replicate', 'Strain', 'Timepoint', 'Norm', 'Plate'])



In [78]:
metadata1[['#','Batch']] = metadata1['#/Batch'].str.split(' / ',n=1, expand=True)

In [79]:
metadata1.Timepoint = [f'T{int(x)}' if x!='NaN' else x for x in metadata1.Timepoint]

In [80]:
column_names = metadata1['#/Batch']
df1.columns = list(df1.columns[:3]) + list(column_names)

In [81]:
metadata1.Strain.unique()

array(['STD', 'satP', 'tsx', 'punC', 'yeaS', 'sstT', 'oppA', 'ybjE',
       'rhtA', 'WT', 'ompF', 'ompC', 'ygaZ', 'yddG', 'yedA', 'mscS',
       'adeP', 'ptsG', 'gltJ', 'aroP', 'nagE', 'codB', 'argO', 'actP',
       'setA', 'brnQ', 'lysP', 'putP', 'acrB', 'yhjE', 'mscL', 'glnP',
       'proP', 'cycA', 'gltP', 'gadC', 'rbsC', 'glpF', 'sdaC', 'znuB',
       'focA', 'artQ', 'fruA', 'nupC', 'cstA', 'mtr', 'gltS', 'ugpA',
       'dcuA', 'uhpT', 'nupG', 'pheP', 'tdcC', 'dctA', 'emrB', 'rhtC',
       'dppB', 'galP', 'yahN', 'WT+pEP65', 'metI',
       'ΔtrpC (sfGFP) proB74 + pEP28', 'E.coli MG1655', 'rhtB', 'manY',
       'livH', 'WT+pEP66_5', 'kgtP', 'eamA', 'ansP', 'WT+pEP35', 'hisQ',
       'glpT', 'ΔproC (mCherry) ΔtrpR + pEP28', 'WT+pEP17', 'tolC'],
      dtype=object)

## Batch 5

In [82]:
metadata5 = pd.DataFrame([[y.strip(' ').replace('?', 'Δ') for y in x.split('|')] for x in df5.columns[4:]], columns = ['Batch', 'Replicate', 'Strain', 'Timepoint', 'Plate', 'Well'])

In [83]:
metadata5['#'] = [str(x).zfill(5) for x in np.arange(1, len(metadata5) + 1)]

In [84]:
metadata5['#/Batch'] = metadata5['#']+'/'+metadata5['Batch']

In [85]:
column_names5 = metadata5['#/Batch']
df5.columns = list(df5.columns[:4]) + list(column_names5)

# Make long DFs

In [284]:
df1L = df1.melt(id_vars=['Metabolite', 'ionMz', 'KEGG'], var_name='Sample ID', value_name='Peak Intensity')
df5L  = df5.iloc[:,1:].melt(id_vars=['Metabolite', 'ionMz', 'kegg'], var_name='Sample ID', value_name='Peak Intensity')


# Add metadata

In [ ]:
df1L = df1L.merge(metadata1, left_on='Sample ID', right_on='#/Batch', how='left')
df5L = df5L.merge(metadata5, left_on='Sample ID', right_on='#/Batch', how='left')



In [286]:
df1L.replace('NaN', np.nan, inplace=True)
df5L.replace('NaN', np.nan, inplace=True)

In [287]:
df1L['Timepoint'] = [float(x.strip('T')) if isinstance(x, str) else x for x in df1L.Timepoint]
df5L['Timepoint'] = df5L['Timepoint'].astype(float)

In [288]:
df5L.rename(columns={'kegg':'KEGG'}, inplace=True)

In [289]:
df1L['Mass-spec run'] = 1
df5L['Mass-spec run'] = 2

In [290]:
df1L.Batch = df1L.Batch.astype(float)
df5L.Batch = df5L.Batch.astype(float)

# Drop batch 5 from df1
and standards after batch 5, plus thw few wrongly annotated samples. Add 10000 to the # in the second run to get unique numbers

In [304]:
df1L = df1L.loc[df1L.Batch != 5]
df1L = df1L.loc[(df1L['#'].astype(int) < 1e4)]
df1L['#'] = df1L['#'].astype(int)
df5L['#'] = df5L['#'].astype(int)
df5L['#'] = df5L['#']+10000

# Subtract instrument drift from batch 1-4

In [292]:
run_drift_correction = False
pi_label = 'Peak Intensity'


In [293]:
if run_drift_correction:
    metabolites = df1L.Metabolite.unique()
    xarr = np.linspace(0, 1e4, 200)
    pi_label = 'Corrected (PI)'
    for i, met in enumerate(metabolites):
        idx = (df1L.Metabolite == met) & (df1L.Timepoint.isin([1,2]))
        x = df1L.loc[idx, '#']
        y = df1L.loc[idx, 'Peak Intensity']
        X = sm.add_constant(x)
        X['x2'] = x**2
        rlm_model = sm.RLM(y,X, M=sm.robust.norms.HuberT())
        rlm_results = rlm_model.fit()
        yarr = rlm_results.params['const'] +xarr*rlm_results.params['#']+xarr**2*rlm_results.params['x2']
        
        idxfit = df1L.Metabolite==met
        x_fit = df1L.loc[idxfit, '#']
        y_all = df1L.loc[idxfit, 'Peak Intensity']
        y_fit = rlm_results.params['const'] + x_fit*rlm_results.params['#']+x_fit**2*rlm_results.params['x2']
        df1L.loc[idxfit, 'Corrected (PI)'] = y_all - y_fit
    # if i > j:
    #     print(met)
    #     break

# plt.plot(x_fit, y_fit)
# plt.plot(x_fit, y_all, 'o', alpha = 0.3)
# plt.plot(x_fit, y_all - y_fit, 'o', alpha = 0.3)

# Remove standards and NaN

In [301]:
df1L = df1L.loc[df1L.Strain != 'STD'].copy()
# df5L = df5L.loc[df5L.Strain != 'STD']
df5L = df5L.loc[df5L.Strain != 'blank'].copy()

df1L.dropna(subset=['Peak Intensity'], inplace=True)
df5L.dropna(subset=['Peak Intensity'], inplace=True)

# Remove outliers
From df1

In [ ]:
def remove_outliers(df, outliers):
    idx_list = []
    for i, outlier_lst in enumerate(outliers):
        
        idxo = (df['#'].isin([int(x) for x in outlier_lst[2]]))&(df.Metabolite==outlier_lst[1])&(df.Strain==outlier_lst[0])
        if np.sum(idxo)==0:
            print('Warning: No outliers found for', outlier_lst)
        if i == 0:
            idx = idxo
        else:
            idx = idx | idxo
    print(f'Remove {np.sum(idx)} datapoints')
    return df.loc[~idx]
    
outliers = [
    ['punC', 'Taurine', ['00029', '00030','00031', '00032']],
    ['punC', 'S-Ribosyl-L-homocysteine', ['00029', '00030','00031', '00032']],
    ['punC', '3-Sulfino-L-alanine', ['00029', '00030','00031', '00032']],
    ['punC', 'Glycine', ['00029', '00030','00031', '00032', '00069', '00070', '00071', '00072']],
    ['punC', 'Pyridoxamine', ['00017', '00018', '00019', '00020', '00029', '00030','00031', '00032', '00033', '00034','00035', '00036']],
    ['punC', 'Acetylornithine', ['01637', '01638', '01639', '01640']],
    ['punC', 'Arginine', ['01637', '01638', '01639', '01640']],
    ['punC', 'Citrulline', ['01637', '01638', '01639', '01640']],
    ['punC', 'Ornithine', ['01637', '01638', '01639', '01640']],
    ['punC', 'IMP', ['01637', '01638', '01639', '01640']],
    ['punC', '(Iso)Citrate', ['01637', '01638', '01639', '01640']],
    ['punC', '7,8-Diaminononanoate', ['01637', '01638', '01639', '01640']],
    ['punC', 'Serine', ['01749', '01750', '01751', '01752', '01125','01126', '01127', '01128']],
    ['punC', 'Histidine', ['01749', '01750', '01751', '01752', '01125','01126', '01127', '01128']],
    
    ['tsx', 'CTP', ['00253', '00254', '00255', '00256']],
    ['tsx', 'D-Ribose 5-diphosphate', ['00253', '00254', '00255', '00256']],
    ['tsx', 'Disaccharide', ['01273', '01274', '01275', '01276']],
    ['tsx', 'Pyridoxamine', ['00013', '00014', '00015', '00016', '00093', '00094','00095', '00096']],
    
    ['artQ', 'FGAR', ['04044', '04045', '04046', '04047']],
    ['artQ', '(E)-4-(Trimethylammonio)but-2-enoate', ['04044', '04045', '04046', '04047']],

    ['argO', 'Glycerol', ['02901', '02902', '02903', '02904']],
    ['codB', 'Glycerol', ['02913', '02914', '02915', '02916']],
    
    ['rhtA', 'Urate', ['00169', '00170', '00171', '00172']],
    ['rhtA', 'Acetylornithine', ['01917', '01918', '01919', '01920']],
    ['rhtA', 'Arginine', ['01917', '01918', '01919', '01920']],
    ['rhtA', 'Hexanoic acid', ['01917', '01918', '01919', '01920']],
    ['rhtA', '3-Dehydroquinate', ['01105', '01106', '01107', '01108']],
    ['rhtA', 'sn-Glycero-3-phosphocholine', ['01105', '01106', '01107', '01108']],
    ['rhtA', 'Phenyl acetate', ['01105', '01106', '01107', '01108']],
    ['rhtA', 'Phosphoglycerate', ['01105', '01106', '01107', '01108']],
    ['rhtA', '(Iso)Citrate', ['01917', '01918', '01919', '01920']],
    
    
    ['ompF', 'Glycine', ['00273', '00274', '00275', '00276']],
    ['ompF', 'Urate', ['00273', '00274', '00275', '00276']],
    ['ompF', 'Phosphoglycerate', ['01077', '01078', '01079', '01080']],

    ['oppA', 'Glutathione disulfide', ['00225', '00226', '00227', '00228']],
    ['oppA', 'Cys', ['00057', '00058']],
    ['oppA', '4-Methyl-2-oxopentanoate', ['01909']],
    ['oppA', 'O-Phospho-4-hydroxy-L-threonine', ['00175']],
    ['oppA', '6-Acetyl-D-glucose', ['01909', '01910']],
    ['oppA', 'N-Acetylneuraminate', ['01909']],
    
    
    ['emrB', 'Histidine', ['06209', '06210']],
    ['emrB', 'Serine', ['06209', '06210']],
    ['emrB', 'Ornithine', ['06209','06210']],
    ['sdaC', 'Serine', ['04940', '04941', '04942', '04943', '04930', '04931']],
    # ['tolC', 'Serine', ['09232', '09233']],
    # ['tolC', 'Histidine', ['09232', '09233']],
    # ['tolC', 'Ornithine', ['09232', '09233']],
    ['nupG', 'Serine', ['06313', '06314']],
    ['nupC', 'Serine', ['04814','04815', '05258', '05259']],
    ['yedA', 'Phenylacetaldehyde', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Phenyl acetate', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', '3-Dehydroquinate', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Ketovaline', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Glutamine', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Glycine', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092', '00261', '00262','00263', '00264']],
    ['yedA', 'N-Acetyl-D-glucosamine', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Met-oxide', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'sn-Glycero-3-phosphocholine', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', '2-Aceto-2-hydroxybutanoate', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'dTDP-hexose', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Deoxyhexose', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Pyridoxamine', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Urate', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Nicotinamide', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Benzyl alcohol', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', '3-Sulfino-L-alanine', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', '(E)-4-(Trimethylammonio)but-2-enoate', ['00213', '00214','00215','00216', '00089', '00090', '00091', '00092']],
    ['yedA', 'Pyridoxine', ['00089', '00090', '00091', '00092']],
    ['yedA', 'Hexanoic acid', ['00089', '00090', '00091', '00092']],
    ['yedA', 'Cytosine', ['00089', '00090', '00091', '00092']],

    ['ybjE', 'Histidine', ['01091', '01092', '00383', '00384']],
    ['ybjE', 'Serine', ['01091', '01092', '00383', '00384']],

    ['yddG', 'FGAR', ['00081', '00082', '00083', '00084']],

    ['yeaS', 'Pyridoxamine', ['00021', '00022', '00023', '00024', '00075', '00076']],
    ['yeaS', 'Phenylacetaldehyde', ['00993', '00994', '00995', '00996']],

    ['ygaZ', 'Histidine', ['00305', '00306', '00307', '00308']],
     

    
    ['satP', 'Pyridoxamine', ['00009', '00010', '00011', '00012']],
    ['satP', 'N-Acetyl-D-glucosamine', ['00009', '00010', '00011', '00012']],
    ['satP', 'Met-oxide', ['00010', '00012']],

    ['mscS','Phosphoglycerate', ['01075','01076']],
    
    ['WT', 'Glycine', ['00293', '00294', '00295', '00296', '00325', '00326', '00327', '00328', '00337', '00338', '00339','00340']],
    ['WT', 'Serine', ['06221', '06222', '01691','01692','04946']],
    ['WT', 'Histidine', ['06221', '06222', '01691','01692','04946']],
    ['WT', 'Succinyl-CoA', ['00313', '00314', '00315','00316']],   
]
df1L = remove_outliers(df1L, outliers)


Remove 413 datapoints


# Normalize

In [309]:
for met in df1L.Metabolite.unique():
    idx = (df1L.Metabolite == met)
    mean = df1L.loc[idx, pi_label].mean()
    std = df1L.loc[idx, pi_label].std()
    df1L.loc[idx, 'Z-score'] = (df1L.loc[idx, pi_label]-mean)/std

for met in df5L.Metabolite.unique():
    idx = (df5L.Metabolite == met)
    mean = df5L.loc[idx, pi_label].mean()
    std = df5L.loc[idx, pi_label].std()
    df5L.loc[idx, 'Z-score'] = (df5L.loc[idx, pi_label]-mean)/std

# Merge

In [311]:
df1L['Mass-spec run'] = 1
df5L['Mass-spec run'] = 2
df = pd.concat([df1L,df5L])

In [332]:
df.drop(columns=['#/Batch', 'Norm'], inplace=True)

# Get OD data

In [325]:
od_fn = metabolomics_folder / 'F_od_and_auc.csv'
od_df = pd.read_csv(od_fn, index_col=0)

In [326]:
strains = sorted([x for x in od_df['Strain'].unique()])

# Merge OD and metabolomics

In [327]:
plate_well_to_medium = defaultdict(lambda: 'M9 20 mM glucose pH 7.4')
temp_dict = samples_df.set_index(['Plate', 'Well'])['Medium'].to_dict()
plate_well_to_medium.update(temp_dict)

### Fix columns before merge

In [328]:
od_df.replace({'Timepoint':'nan'}, np.nan, inplace=True)
od_df['Timepoint'] = [float(x.strip('T')) if isinstance(x, str) else x for x in od_df.Timepoint]


In [334]:
df['Well'] = [x.split('.')[0] if isinstance(x, str) else x for x in df['Well']]


In [335]:
df['Medium'] = df.apply(lambda x: plate_well_to_medium[x['Plate'], x['Well']], axis=1)


In [337]:
df_full = pd.merge(left = df, right = od_df, left_on=('Strain','Replicate','Timepoint', 'Batch', 'Medium'), right_on=('Strain', 'Parallel', 'Timepoint', 'Batch', 'Medium'), how='left')



# Get median of the four technical replicates

In [343]:
df_full.head()

,Metabolite,ionMz,KEGG,Sample ID,Peak Intensity,Replicate,Strain,Timepoint,Plate,#,...,Medium,Strain #,Tube #,Parallel,Tube,Hours,OD,Batch-Tube,Exp. phase time limit,AUC OD
0,Acetone,57.0346,C00207; C00479; C02001; C11506; C11507; C15508,00009 / 1,16653,A,satP,1.0,P1,9,...,M9 20 mM glucose pH 7.4,7,7.0,A,7A,4.0,0.025,1-7A,13.0,0.064655
1,Acetatic acid,59.0135,C00033; C00266,00009 / 1,253609,A,satP,1.0,P1,9,...,M9 20 mM glucose pH 7.4,7,7.0,A,7A,4.0,0.025,1-7A,13.0,0.064655
2,Propenoic acid C3:1,71.0141,C00511; C00546; C19246; C19297; C19440,00009 / 1,114935,A,satP,1.0,P1,9,...,M9 20 mM glucose pH 7.4,7,7.0,A,7A,4.0,0.025,1-7A,13.0,0.064655
3,Iminoacetate,72.0093,C15809,00009 / 1,804,A,satP,1.0,P1,9,...,M9 20 mM glucose pH 7.4,7,7.0,A,7A,4.0,0.025,1-7A,13.0,0.064655
4,Glyoxylic acid,72.9932,C00048,00009 / 1,2855,A,satP,1.0,P1,9,...,M9 20 mM glucose pH 7.4,7,7.0,A,7A,4.0,0.025,1-7A,13.0,0.064655


In [356]:
df_average = df_full.groupby(['Batch-Tube', 'Timepoint', 'ionMz', 'Medium']).agg({'Z-score':('mean','std','median'),'OD':'first', 'AUC OD':'first', 'Hours':'first', 'Strain':'first', 
                                                                    'Parallel':'first', 'Batch':'first', 
                                                                    '#':'first', 'Exp. phase time limit':'first', 'Tube':'first','Well':'first', 'Metabolite':'first', 'KEGG':'first'}).reset_index()


In [357]:
df_average.columns = ['Batch-Tube', 'Timepoint', 'ionMz', 'Medium', 'Z-score mean', 'Z-score std', 'Z-score median', 'OD', 'AUC OD', 'Hours', 'Strain','Parallel', 'Batch', '#', 'Exp. phase time limit', 'Tube', 'Well', 'Metabolite', 'KEGG']

# Save median values

In [358]:
df_median = df_average[['Batch-Tube', 'Timepoint', 'ionMz', 'Z-score median']]
df_median.to_csv(metabolomics_folder / f'F_processed_median_Z-scores_{normalization}_norm.csv', index=False)

## Save without metadata to reduce space

In [359]:
fn_median = metabolomics_folder / f'G_median_z_scores_{normalization}_norm.csv'
df_median.to_csv(fn_median)

# Save metadata
ionMZ and sample metdata seperately

In [360]:
mz_annotation_df = df_average[['ionMz', 'Metabolite', 'KEGG']].drop_duplicates().reset_index(drop=True)

In [362]:
mz_annotation_df.to_csv(metabolomics_folder / f'H_ionMz_annotation.csv', index=False)

In [363]:
sample_metadata = df_average[['Batch-Tube', 'Timepoint', 'Medium', 'OD', 'AUC OD', 'Hours', 'Strain','Parallel', 'Batch', '#', 'Exp. phase time limit', 'Tube', 'Well']].drop_duplicates().reset_index(drop=True)

In [365]:
sample_metadata.to_csv(metabolomics_folder / f'I_sample_metadata.csv', index=False)